# Building a SQL Agent with LangChain

An agent that answers plain-English questions about a SQL database: it inspects the
schema, writes a query, validates it, runs it, and explains the result.

The data is [Chinook](https://www.sqlitetutorial.net/sqlite-sample-database/), a
SQLite sample database for a digital media store.

## Setup

Dependencies live in `requirements.txt` and are installed into the project's `.venv`:

```bash
uv venv --python 3.12 .venv
uv pip install --python .venv/bin/python -r requirements.txt
```

Versions are pinned. `create_sql_agent(agent_type="openai-tools")` is part of the
`langchain-community` 0.3.x line and was reorganised in later releases.

Select the **Python (langchain-sql-agent)** kernel before running.

In [1]:
import importlib.metadata as md

for pkg in ["langchain", "langchain-community", "langchain-openai", "langsmith"]:
    print(f"{pkg:22} {md.version(pkg)}")

langchain              0.3.4
langchain-community    0.3.3
langchain-openai       0.2.2
langsmith              0.1.147


## Configuration

Keys are read from the `.env` file in this folder, which is gitignored.

```
OPENAI_API_KEY=sk-...
```

LangSmith tracing is optional. Enable it with `LANGSMITH_TRACING=true` and a
`LANGSMITH_API_KEY` to see each agent step, tool call, and token count at
[smith.langchain.com](https://smith.langchain.com).

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to .env and re-run this cell.")

tracing = os.getenv("LANGSMITH_TRACING", "false").lower() == "true" and os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true" if tracing else "false"

print("OpenAI key loaded")
print("LangSmith tracing:", "on" if tracing else "off")

OpenAI key loaded
LangSmith tracing: off


## The database

Download `Chinook.db` once, then reuse the local copy.

In [3]:
import pathlib

import requests

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
local_path = pathlib.Path("Chinook.db")

if local_path.exists():
    print(f"{local_path} already present ({local_path.stat().st_size:,} bytes)")
else:
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    local_path.write_bytes(response.content)
    print(f"Downloaded {local_path} ({local_path.stat().st_size:,} bytes)")

Chinook.db already present (913,408 bytes)


`SQLDatabase` is LangChain's wrapper around a SQLAlchemy connection. On its own it is
just a convenience layer for running SQL from Python — no model involved yet. It also
becomes the thing the agent's tools read from.

In [4]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Tables:  {db.get_usable_table_names()}")

Dialect: sqlite
Tables:  ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [5]:
print(db.run("SELECT * FROM Artist LIMIT 5;"))

[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


In [6]:
print(db.get_table_info(["Genre"]))


CREATE TABLE "Genre" (
	"GenreId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("GenreId")
)

/*
3 rows from Genre table:
GenreId	Name
1	Rock
2	Jazz
3	Metal
*/


## The model

The agent needs a model that supports tool calling, since every database action is a
tool call rather than free-form text.

In [7]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1", temperature=0)
model.model_name

'gpt-4.1'

## Building the agent

`create_sql_agent` bundles the database into four tools and wires them to the model:

| Tool | Purpose |
| --- | --- |
| `sql_db_list_tables` | List the available tables |
| `sql_db_schema` | Show a table's schema plus sample rows |
| `sql_db_query_checker` | Review the SQL before it runs |
| `sql_db_query` | Execute the query |

`agent_type="openai-tools"` uses OpenAI function calling, so the model picks a tool
against a strict JSON schema instead of reasoning its way through a text prompt the way
the older ReAct agent did. Fewer malformed queries, and the tool arguments are typed.

`verbose=True` prints each step, which is what makes the reasoning visible below.

In [8]:
from langchain_community.agent_toolkits import create_sql_agent

agent = create_sql_agent(
    llm=model,
    db=db,
    agent_type="openai-tools",
    verbose=True,
)

[tool.name for tool in agent.tools]

['sql_db_query', 'sql_db_schema', 'sql_db_list_tables', 'sql_db_query_checker']

## Asking questions

Each call runs the full loop: list tables, read the relevant schema, check the SQL,
execute it, then answer in plain English.

In [9]:
agent.invoke({"input": "How many tracks are in the Chinook database?"})



> Entering new SQL Agent Executor chain...



Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track


Invoking: `sql_db_schema` with `{'table_names': 'Track'}`



CREATE TABLE "Track" (
	"TrackId" INTEGER NOT NULL, 
	"Name" NVARCHAR(200) NOT NULL, 
	"AlbumId" INTEGER, 
	"MediaTypeId" INTEGER NOT NULL, 
	"GenreId" INTEGER, 
	"Composer" NVARCHAR(220), 
	"Milliseconds" INTEGER NOT NULL, 
	"Bytes" INTEGER, 
	"UnitPrice" NUMERIC(10, 2) NOT NULL, 
	PRIMARY KEY ("TrackId"), 
	FOREIGN KEY("MediaTypeId") REFERENCES "MediaType" ("MediaTypeId"), 
	FOREIGN KEY("GenreId") REFERENCES "Genre" ("GenreId"), 
	FOREIGN KEY("AlbumId") REFERENCES "Album" ("AlbumId")
)

/*
3 rows from Track table:
TrackId	Name	AlbumId	MediaTypeId	GenreId	Composer	Milliseconds	Bytes	UnitPrice
1	For Those About To Rock (We Salute You)	1	1	1	Angus Young, Malcolm Young, Brian Johnson	343719	11170334	0.99
2	Balls to the Wall	2	2	1	None	342562	5510424	0.99
3	Fast As a Shark	3	2	1	F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman	230619	3990994	0.99
*/


Invoking: `sql_db_query_checker` with `{'query': 'SELECT COUNT(*) AS track_count FROM Track;'}`




```sql
SELECT COUNT(*) AS track_count FROM Track;
```


Invoking: `sql_db_query` with `{'query': 'SELECT COUNT(*) AS track_count FROM Track;'}`


[(3503,)]

There are 3,503 tracks in the Chinook database.

> Finished chain.


{'input': 'How many tracks are in the Chinook database?',
 'output': 'There are 3,503 tracks in the Chinook database.'}

In [10]:
agent.invoke({"input": "List 5 customers from India."})



> Entering new SQL Agent Executor chain...



Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track


Invoking: `sql_db_schema` with `{'table_names': 'Customer'}`



CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Customer table:
CustomerId	FirstName	LastName	Company	Address	City	State	Country	PostalCode	Phone	Fax	Email	SupportRepId
1	Luís	Gonçalves	Embraer - Empresa Brasileira de Aeronáutica S.A.	Av. Brigadeiro Faria Lima, 2170	São José dos Campos	SP	Brazil	12227-000	+55 (12) 3923-5555	+55 (12) 3923-5566	luisg@embraer.com.br	3
2	Leonie	Köhler	None	Theodor-Heuss-Straße 34	Stuttgart	None	Germany	70174	+49 0711 2842222	None	leonekohler@su


Invoking: `sql_db_query_checker` with `{'query': "SELECT FirstName, LastName, City, Email FROM Customer WHERE Country = 'India' LIMIT 5;"}`




SELECT FirstName, LastName, City, Email FROM Customer WHERE Country = 'India' LIMIT 5;


Invoking: `sql_db_query` with `{'query': "SELECT FirstName, LastName, City, Email FROM Customer WHERE Country = 'India' LIMIT 5;"}`


[('Manoj', 'Pareek', 'Delhi', 'manoj.pareek@rediff.com'), ('Puja', 'Srivastava', 'Bangalore', 'puja_srivastava@yahoo.in')]

Here are customers from India (up to 5):

1. Manoj Pareek, Delhi, manoj.pareek@rediff.com
2. Puja Srivastava, Bangalore, puja_srivastava@yahoo.in

There are only 2 customers from India in the database.

> Finished chain.


{'input': 'List 5 customers from India.',
 'output': 'Here are customers from India (up to 5):\n\n1. Manoj Pareek, Delhi, manoj.pareek@rediff.com\n2. Puja Srivastava, Bangalore, puja_srivastava@yahoo.in\n\nThere are only 2 customers from India in the database.'}

A question that needs a join and an aggregate, rather than a single-table lookup.

In [11]:
agent.invoke({"input": "Which genre has the longest tracks on average? Give the top 3."})



> Entering new SQL Agent Executor chain...



Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track


Invoking: `sql_db_schema` with `{'table_names': 'Genre, Track'}`



CREATE TABLE "Genre" (
	"GenreId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("GenreId")
)

/*
3 rows from Genre table:
GenreId	Name
1	Rock
2	Jazz
3	Metal
*/


CREATE TABLE "Track" (
	"TrackId" INTEGER NOT NULL, 
	"Name" NVARCHAR(200) NOT NULL, 
	"AlbumId" INTEGER, 
	"MediaTypeId" INTEGER NOT NULL, 
	"GenreId" INTEGER, 
	"Composer" NVARCHAR(220), 
	"Milliseconds" INTEGER NOT NULL, 
	"Bytes" INTEGER, 
	"UnitPrice" NUMERIC(10, 2) NOT NULL, 
	PRIMARY KEY ("TrackId"), 
	FOREIGN KEY("MediaTypeId") REFERENCES "MediaType" ("MediaTypeId"), 
	FOREIGN KEY("GenreId") REFERENCES "Genre" ("GenreId"), 
	FOREIGN KEY("AlbumId") REFERENCES "Album" ("AlbumId")
)

/*
3 rows from Track table:
TrackId	Name	AlbumId	MediaTypeId	GenreId	Composer	Milliseconds	Bytes	UnitPrice
1	For Those About To Rock (We Salute You)	1	1	1	Angus Young, Malcolm Young, Brian Johnson	343719	11170334	0.99
2	Balls to the Wall	2	2	1	None	342562	5510424	


Invoking: `sql_db_query_checker` with `{'query': 'SELECT Genre.Name, AVG(Track.Milliseconds) AS AvgTrackLength\nFROM Track\nJOIN Genre ON Track.GenreId = Genre.GenreId\nGROUP BY Genre.GenreId\nORDER BY AvgTrackLength DESC\nLIMIT 3;'}`




```sql
SELECT Genre.Name, AVG(Track.Milliseconds) AS AvgTrackLength
FROM Track
JOIN Genre ON Track.GenreId = Genre.GenreId
GROUP BY Genre.GenreId
ORDER BY AvgTrackLength DESC
LIMIT 3;
```


Invoking: `sql_db_query` with `{'query': 'SELECT Genre.Name, AVG(Track.Milliseconds) AS AvgTrackLength\nFROM Track\nJOIN Genre ON Track.GenreId = Genre.GenreId\nGROUP BY Genre.GenreId\nORDER BY AvgTrackLength DESC\nLIMIT 3;'}`


[('Sci Fi & Fantasy', 2911783.0384615385), ('Science Fiction', 2625549.076923077), ('Drama', 2575283.78125)]

The top 3 genres with the longest tracks on average are:
1. Sci Fi & Fantasy
2. Science Fiction
3. Drama

> Finished chain.


{'input': 'Which genre has the longest tracks on average? Give the top 3.',
 'output': 'The top 3 genres with the longest tracks on average are:\n1. Sci Fi & Fantasy\n2. Science Fiction\n3. Drama'}

And one that spans three tables.

In [12]:
agent.invoke({"input": "Which country generated the most revenue? Show the top 5 with totals."})



> Entering new SQL Agent Executor chain...



Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track


Invoking: `sql_db_schema` with `{'table_names': 'Invoice, Customer'}`



CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Customer table:
CustomerId	FirstName	LastName	Company	Address	City	State	Country	PostalCode	Phone	Fax	Email	SupportRepId
1	Luís	Gonçalves	Embraer - Empresa Brasileira de Aeronáutica S.A.	Av. Brigadeiro Faria Lima, 2170	São José dos Campos	SP	Brazil	12227-000	+55 (12) 3923-5555	+55 (12) 3923-5566	luisg@embraer.com.br	3
2	Leonie	Köhler	None	Theodor-Heuss-Straße 34	Stuttgart	None	Germany	70174	+49 0711 2842222	None	leone


Invoking: `sql_db_query_checker` with `{'query': 'SELECT BillingCountry, SUM(Total) as TotalRevenue\nFROM Invoice\nGROUP BY BillingCountry\nORDER BY TotalRevenue DESC\nLIMIT 5;'}`




```sql
SELECT BillingCountry, SUM(Total) as TotalRevenue
FROM Invoice
GROUP BY BillingCountry
ORDER BY TotalRevenue DESC
LIMIT 5;
```


Invoking: `sql_db_query` with `{'query': 'SELECT BillingCountry, SUM(Total) as TotalRevenue\nFROM Invoice\nGROUP BY BillingCountry\nORDER BY TotalRevenue DESC\nLIMIT 5;'}`


[('USA', 523.06), ('Canada', 303.96), ('France', 195.1), ('Brazil', 190.1), ('Germany', 156.48)]

The top 5 countries that generated the most revenue are:

1. USA - $523.06
2. Canada - $303.96
3. France - $195.10
4. Brazil - $190.10
5. Germany - $156.48

> Finished chain.


{'input': 'Which country generated the most revenue? Show the top 5 with totals.',
 'output': 'The top 5 countries that generated the most revenue are:\n\n1. USA - $523.06\n2. Canada - $303.96\n3. France - $195.10\n4. Brazil - $190.10\n5. Germany - $156.48'}

The default system prompt tells the agent not to run `INSERT`, `UPDATE`, `DELETE` or
`DROP`. Asked to do so anyway, it stops before calling a single tool and falls back to
the prompt's "I don't know" response — terse, but nothing reached the database.

In [13]:
agent.invoke({"input": "Delete all rows from the Album table."})



> Entering new SQL Agent Executor chain...


I don't know.

> Finished chain.


{'input': 'Delete all rows from the Album table.', 'output': "I don't know."}

In [14]:
print(db.run("SELECT COUNT(*) FROM Album;"))

[(347,)]


This is a prompt-level guardrail, not a database-level one. For anything beyond a demo,
connect as a read-only user so the permission is enforced by the database itself.

## Where to take this

- Point `SQLDatabase.from_uri` at Postgres or MySQL instead of SQLite
- Wrap it in a UI — `sql_app.py` and `sql_agent_app.py` in this repo do that with Streamlit
- Turn on LangSmith tracing to see cost and latency per step
- Restrict the connection to a read-only database user, rather than relying on the prompt

## References

- [LangChain SQL agent guide](https://docs.langchain.com/oss/python/langchain/sql-agent)
- [LangChain agents](https://docs.langchain.com/oss/python/langchain/agents)
- [Chinook sample database](https://www.sqlitetutorial.net/sqlite-sample-database/)